# example of reading one trace

In [1]:
import Pkg

Pkg.instantiate()

using Pkg

cd(@__DIR__)
Pkg.activate("../")
Pkg.add(url="https://github.com/tp2750/TuningSystems.jl") 

Pkg.add("DSP")
using DSP
using Dates, SonifSismo, CairoMakie
using Statistics
using WAV
using Seis
using DSP: conv
using TuningSystems
using Test

archive = DataArchive("C:/Users/lucie/Desktop/mtFujiContinuous/mtFujiContinuous")
push!(LOAD_PATH, "C:/Users/lucie/Desktop/TuningSystems.jl-main/TuningSystems.jl-main/src")
include("C:/Users/lucie/Desktop/sonifSismo.jl-main/examples/sonify_one_trace.jl")

  Activating project at `c:\Users\lucie\Desktop\sonifSismo.jl-main`
    Updating git-repo `https://github.com/tp2750/TuningSystems.jl`
   Resolving package versions...
      Compat entries added for 
     Project No packages added to or removed from `C:\Users\lucie\Desktop\sonifSismo.jl-main\Project.toml`
    Manifest No packages added to or removed from `C:\Users\lucie\Desktop\sonifSismo.jl-main\Manifest.toml`
   Resolving package versions...
      Compat entries added for 
     Project No packages added to or removed from `C:\Users\lucie\Desktop\sonifSismo.jl-main\Project.toml`
    Manifest No packages added to or removed from `C:\Users\lucie\Desktop\sonifSismo.jl-main\Manifest.toml`


false

In [2]:
available_stations(archive)
available_channels(archive)

33-element Vector{ChannelAvailability}:
 ChannelAvailability("EV", "FJO", "", "E", [Date("2008-05-25"), Date("2008-05-26"), Date("2008-05-27"), Date("2008-05-28"), Date("2008-05-29"), Date("2008-05-30"), Date("2008-05-31")], ["C:\\Users\\lucie\\Desktop\\mtFujiContinuous\\mtFujiContinuous\\mseed\\2008\\05\\25\\EV.FJO..E.20080525.mseed", "C:\\Users\\lucie\\Desktop\\mtFujiContinuous\\mtFujiContinuous\\mseed\\2008\\05\\26\\EV.FJO..E.20080526.mseed", "C:\\Users\\lucie\\Desktop\\mtFujiContinuous\\mtFujiContinuous\\mseed\\2008\\05\\27\\EV.FJO..E.20080527.mseed", "C:\\Users\\lucie\\Desktop\\mtFujiContinuous\\mtFujiContinuous\\mseed\\2008\\05\\28\\EV.FJO..E.20080528.mseed", "C:\\Users\\lucie\\Desktop\\mtFujiContinuous\\mtFujiContinuous\\mseed\\2008\\05\\29\\EV.FJO..E.20080529.mseed", "C:\\Users\\lucie\\Desktop\\mtFujiContinuous\\mtFujiContinuous\\mseed\\2008\\05\\30\\EV.FJO..E.20080530.mseed", "C:\\Users\\lucie\\Desktop\\mtFujiContinuous\\mtFujiContinuous\\mseed\\2008\\05\\31\\EV.FJO..E.2008053

In [3]:
start_time = DateTime(2008, 5, 28, 0, 0, 0)  # UTC
end_time   = DateTime(2008, 5, 28, 23, 59, 59)     # UTC

traces = read_window(
    archive,
    start_time,
    end_time;
    stations="FUJ",
    channels=["wE", "wN", "wU"],
    merge=true,
 #    merge_gaps=:zero,       # or :linear
    processed=false,
    bandpass=(0.5, 15.0),
)

3-element Vector{AbstractTrace}:
 Seis.Trace(EV.FUJ..wE : delta=0.01, b=0.0, nsamples=8639901)
 Seis.Trace(EV.FUJ..wN : delta=0.01, b=0.0, nsamples=8639901)
 Seis.Trace(EV.FUJ..wU : delta=0.01, b=0.0, nsamples=8639901)

In [4]:
trace_wU = only(t for t in traces if strip(t.sta.cha) == "wU")
trace_wN = only(t for t in traces if strip(t.sta.cha) == "wN")
trace_wE = only(t for t in traces if strip(t.sta.cha) == "wE")


Seis.Trace{Float64,Vector{Float32},Seis.Geographic{Float64}}:
            b: 0.0
        delta: 0.01
 GeogStation{Float64}:
      sta.net: EV
      sta.sta: FUJ
      sta.loc: 
      sta.cha: wE 
     sta.meta: Seis.SeisDict{Symbol, Any}()
 GeogEvent{Float64}:
     evt.time: 2008-05-28T00:00:00
     evt.meta: Seis.SeisDict{Symbol, Any}()
 Trace:
        picks: 0
         meta: mseed_file => "C:\\Users\\lucie\\Desktop\\mtFujiContinuous\\mtFujiContinuous\\mseed\\2008\\05\\28\\EV.FUJ..wE.20080528.mseed"

In [40]:
# Construction des 3 signaux, un par un
audioE = sonify_trace(               
    trace_wE;
    acceleration=128,
    gain=4,
    saturation=:soft,
)

audioN = sonify_trace(
    trace_wN;
    acceleration=128,
    gain=4,
    saturation=:soft,
)

audioU = sonify_trace(
    trace_wU;
    acceleration=128,
    gain=4,
    saturation=:soft,
)

#= Extraction des signaux
sigE = audioE.signal     # On est obligé de faire ça pour récupérer les signaux, car les objets audio sont des tuples (contiennent d'autres informations comme la fréquence d'échantillonnage, etc.)
sigN = audioN.signal
sigU = audioU.signal

# Vérification longueur des signaux (car tous les signaux ne sont pas forcément de la même longueur, à cause des éventuels trous dans les données d'origine)
m = minimum(length.([sigE, sigN, sigU]))

sigE = sigE[1:m]
sigN = sigN[1:m]
sigU = sigU[1:m]

# Combinaison des 3 signaux en un seul
sig = (sigE .+ sigN .+ sigU) ./ 3

sig ./= maximum(abs.(sig))

# Enregistrement du signal combiné dans un fichier WAV
wavwrite(
    sig,
    "audio/seisme_3c_Test.wav";
    Fs=audioE.sample_rate     # Fs = fréquence d'échantillonnage de sortie (la fonction de sonification a déjà choisi une fréquence adaptée)
)


Base.Meta.ParseError: ParseError:
# Error @ c:\Users\lucie\Desktop\sonifSismo.jl-main\notebooks\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_W5sZmlsZQ==.jl:23:1

┌────────────────────────
#= Extraction des signaux
sigE = audioE.signal     # On est obligé de faire ça pour récupérer les signaux, car les objets audio sont des tuples (contiennent d'autres informations comme la fréquence d'échantillonnage, etc.)
⋮
    Fs=audioE.sample_rate     # Fs = fréquence d'échantillonnage de sortie (la fonction de sonification a déjà choisi une fréquence adaptée)
)
┘ ── unterminated multi-line comment #= ... =#

In [32]:
Do = s(t.(n.("C")))
save_wav("audio/Do.wav", Do)

Re = s(t.(n.("D")))
save_wav("audio/Re.wav", Re)

Mi = s(t.(n.("E")))
save_wav("audio/Mi.wav", Mi)

In [34]:
son_N = convolve_with_audio(audioN,"C:/Users/lucie/Desktop/sonifSismo.jl-main/notebooks/audio/Do.wav"; kernel_seconds=2, mix=0.35)
son_U = convolve_with_audio(audioU,"C:/Users/lucie/Desktop/sonifSismo.jl-main/notebooks/audio/Re.wav"; kernel_seconds=2, mix=0.35)
son_E = convolve_with_audio(audioE,"C:/Users/lucie/Desktop/sonifSismo.jl-main/notebooks/audio/Mi.wav"; kernel_seconds=2, mix=0.35)

(signal = [-0.2425935420695543, -0.24263764917753056, -0.24268584156448128, -0.24273813066728375, -0.24279528003171313, -0.24285767920330298, -0.24292488560934608, -0.2429958711991845, -0.24306172754785813, -0.24312971105960615  …  7.680744465371582e-5, 7.642157538042408e-5, 7.603188545488105e-5, 7.565077641452144e-5, 7.529040892552241e-5, 7.496264952874964e-5, 7.467901926009338e-5, 7.445064250182852e-5, 7.428819950385452e-5, 7.42018818103606e-5], sample_rate = 44100, playback_factor = 128.0, seismic_duration = 86399.01, audio_duration = 675.9922448979592)

In [38]:
m = minimum([
    length(son_E.signal),
    length(son_N.signal),
    length(son_U.signal)
])

son = (
    son_E.signal[1:m] .+
    son_N.signal[1:m] .+
    son_U.signal[1:m]
) ./ 3

wavwrite(
    son,
    "audio/Musique_conv.wav";
    Fs=audioE.sample_rate     # Fs = fréquence d'échantillonnage de sortie (la fonction de sonification a déjà choisi une fréquence adaptée)
)